# Extend the Daily Beijing Pollutant Dataset

This notebook converts the Beijing hourly pollutant readings into daily summaries, aligns them with the existing pollutant-only daily CSV, and saves a separate expanded pollutant dataset.

It writes:

1. `CNEMC_Figshare_2014-2025/data/beijing_project_stations_daily.csv` — one daily record per station from the newer source, with valid-hour counts.
2. `air-quality-classification/data/beijing_daily_pollutants_combined_2013-2025.csv` — the old and new daily pollutant records appended into the same eight-column format.

The daily bins use local China Standard Time dates. PM2.5, PM10, SO2, NO2, and CO are 24-hour means requiring at least 20 valid hourly readings. O3 is the daily maximum trailing 8-hour mean. CO stays in mg/m³; the other pollutants stay in µg/m³. Where both sources contain the same station-day, the existing row is kept. The old CSV is never overwritten.

## 1. Set up paths and aggregation rules

This cell finds the project folder from either the repository root or this folder, identifies both input files, and defines where the new daily and appended CSVs will be written. It also sets the daily pollutant completeness rules.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_FOLDER = "CNEMC_Figshare_2014-2025"
current_folder = Path.cwd().resolve()
project_candidates = [
    current_folder,
    current_folder / PROJECT_FOLDER,
    current_folder.parent / PROJECT_FOLDER,
]
PROJECT_DIR = next(
    (
        candidate
        for candidate in project_candidates
        if candidate.name == PROJECT_FOLDER and (candidate / "data").is_dir()
    ),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        f"Could not find {PROJECT_FOLDER}/data from the notebook's working directory: {current_folder}"
    )

REPO_ROOT = PROJECT_DIR.parent
DATA_DIR = PROJECT_DIR / "data"
POLLUTANT_INPUT = DATA_DIR / "beijing_project_stations_hourly.csv"
DAILY_POLLUTANT_OUTPUT = DATA_DIR / "beijing_project_stations_daily.csv"
LEGACY_DAILY_INPUT = REPO_ROOT / "air-quality-classification" / "data" / "aqi_model_dataset.csv"
COMBINED_POLLUTANTS_OUTPUT = (
    REPO_ROOT
    / "air-quality-classification"
    / "data"
    / "beijing_daily_pollutants_combined_2013-2025.csv"
)

DAILY_MEAN_POLLUTANTS = ["PM2.5", "PM10", "SO2", "NO2", "CO"]
MIN_VALID_HOURS = 20
MIN_OZONE_WINDOW_READINGS = 6
MIN_OZONE_WINDOWS_PER_DAY = 14
OZONE_WINDOW_END_HOURS = (8, 23)

for input_path in [POLLUTANT_INPUT, LEGACY_DAILY_INPUT]:
    if not input_path.is_file():
        raise FileNotFoundError(f"Required input file is missing: {input_path}")

print(f"Project folder: {PROJECT_DIR}")
print(f"Hourly pollutant input: {POLLUTANT_INPUT.name}")
print(f"Legacy daily pollutant input: {LEGACY_DAILY_INPUT.name}")
print(f"New daily pollutant output: {DAILY_POLLUTANT_OUTPUT.name}")
print(f"Combined pollutant output: {COMBINED_POLLUTANTS_OUTPUT.name}")

Project folder: C:\Users\PLP\Desktop\DataScienceProjects\Data_Science\CNEMC_Figshare_2014-2025
Hourly pollutant input: beijing_project_stations_hourly.csv
Legacy daily pollutant input: aqi_model_dataset.csv
New daily pollutant output: beijing_project_stations_daily.csv
Combined pollutant output: beijing_daily_pollutants_combined_2013-2025.csv


## 2. Load and validate hourly pollutant readings

This cell reads the hourly Beijing CSV, parses its `+08:00` timestamps, and derives a local calendar date for every row. It checks that station-hour records are unique. Negative pollutant values are treated as missing, following the project’s daily AQI preparation rules.

In [2]:
POLLUTANTS = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
required_pollutant_columns = [
    "timestamp", "station_id", "station_name", "station_name_zh", *POLLUTANTS
]

hourly = pd.read_csv(
    POLLUTANT_INPUT,
    dtype={"station_id": "string"},
    low_memory=False,
)
missing_columns = [column for column in required_pollutant_columns if column not in hourly.columns]
if missing_columns:
    raise ValueError(f"Pollutant input is missing required columns: {missing_columns}")

hourly["timestamp"] = pd.to_datetime(hourly["timestamp"], errors="coerce")
if hourly["timestamp"].isna().any():
    bad_count = int(hourly["timestamp"].isna().sum())
    raise ValueError(f"Found {bad_count} rows with an invalid timestamp.")
if hourly["station_id"].isna().any():
    raise ValueError("Some pollutant rows have no station_id.")

# The timestamps carry China Standard Time (+08:00); using .dt.date keeps that local date.
hourly["date"] = hourly["timestamp"].dt.strftime("%Y-%m-%d")
if hourly.duplicated(["station_id", "timestamp"]).any():
    raise ValueError("Duplicate station-hour records found; resolve them before daily aggregation.")

for pollutant in POLLUTANTS:
    values = pd.to_numeric(hourly[pollutant], errors="coerce")
    hourly[pollutant] = values.mask(values < 0)

hourly = hourly.sort_values(["station_id", "timestamp"]).reset_index(drop=True)
station_day_rows = hourly.groupby(["station_id", "date"], sort=True).size()
print(f"Hourly rows: {len(hourly):,}")
print(f"Stations: {hourly['station_id'].nunique()}")
print(f"Date range: {hourly['date'].min()} to {hourly['date'].max()}")
print(f"Rows present per station-day (count: number of station-days): {station_day_rows.value_counts().sort_index().to_dict()}")

Hourly rows: 1,120,320
Stations: 12
Date range: 2014-05-13 to 2025-01-04
Rows present per station-day (count: number of station-days): {24: 46680}


## 3. Aggregate the hourly readings by station-day

This cell creates one row per station and date. It calculates the daily means for PM2.5, PM10, SO2, NO2, and CO when at least 20 valid hourly readings are available. For O3, it calculates trailing 8-hour averages and keeps each day’s maximum when at least 14 valid windows are available. Valid-hour counts remain in this new daily CSV for quality review.

In [3]:
aggregation = {
    "station_name": ("station_name", "first"),
    "station_name_zh": ("station_name_zh", "first"),
    "hours_present": ("timestamp", "nunique"),
}
for pollutant in DAILY_MEAN_POLLUTANTS:
    aggregation[f"{pollutant}_mean"] = (pollutant, "mean")
    aggregation[f"{pollutant}_valid_hours"] = (pollutant, "count")
aggregation["O3_valid_hours"] = ("O3", "count")
aggregation["O3_daily_max_1h"] = ("O3", "max")

daily = (
    hourly.groupby(["station_id", "date"], sort=True)
    .agg(**aggregation)
    .reset_index()
)
for pollutant in DAILY_MEAN_POLLUTANTS:
    daily[f"{pollutant}_24h_avg"] = daily[f"{pollutant}_mean"].where(
        daily[f"{pollutant}_valid_hours"].ge(MIN_VALID_HOURS)
    )
    daily = daily.drop(columns=[f"{pollutant}_mean"])

# Calculate the trailing 8-hour O3 window separately for each station.
o3_daily_parts = []
for station_id, station_rows in hourly.groupby("station_id", sort=True):
    ozone = station_rows.set_index("timestamp").sort_index()["O3"]
    rolling_8h = ozone.rolling(
        "8h",
        closed="right",
        min_periods=MIN_OZONE_WINDOW_READINGS,
    ).mean()
    end_hour = rolling_8h.index.hour
    eligible_windows = rolling_8h.loc[
        (end_hour >= OZONE_WINDOW_END_HOURS[0])
        & (end_hour <= OZONE_WINDOW_END_HOURS[1])
    ]
    window_stats = eligible_windows.groupby(eligible_windows.index.normalize()).agg(["max", "count"])
    if window_stats.empty:
        continue

    window_stats["date"] = window_stats.index.strftime("%Y-%m-%d")
    window_stats["station_id"] = station_id
    window_stats["O3_daily_max_8h_avg"] = window_stats["max"].where(
        window_stats["count"].ge(MIN_OZONE_WINDOWS_PER_DAY)
    )
    o3_daily_parts.append(
        window_stats[["station_id", "date", "O3_daily_max_8h_avg", "count"]]
        .rename(columns={"count": "O3_8h_valid_windows"})
        .reset_index(drop=True)
    )

if o3_daily_parts:
    o3_daily = pd.concat(o3_daily_parts, ignore_index=True)
else:
    o3_daily = pd.DataFrame(
        columns=["station_id", "date", "O3_daily_max_8h_avg", "O3_8h_valid_windows"]
    )
daily = daily.merge(
    o3_daily,
    on=["station_id", "date"],
    how="left",
    validate="one_to_one",
)
daily["O3_8h_valid_windows"] = daily["O3_8h_valid_windows"].fillna(0).astype("int64")

daily_columns = ["date", "station_id", "station_name", "station_name_zh", "hours_present"]
for pollutant in DAILY_MEAN_POLLUTANTS:
    daily_columns.extend([f"{pollutant}_24h_avg", f"{pollutant}_valid_hours"])
daily_columns.extend(
    ["O3_daily_max_8h_avg", "O3_8h_valid_windows", "O3_daily_max_1h", "O3_valid_hours"]
)
daily = daily[daily_columns].sort_values(["station_id", "date"]).reset_index(drop=True)

print(f"Daily station-day rows: {len(daily):,}")
print(f"Daily date range: {daily['date'].min()} to {daily['date'].max()}")
print("Daily means available after the 20-hour rule:")
for pollutant in DAILY_MEAN_POLLUTANTS:
    available = int(daily[f"{pollutant}_24h_avg"].notna().sum())
    print(f"  {pollutant}: {available:,} / {len(daily):,} station-days")
print(f"O3 daily 8-hour maximum available: {daily['O3_daily_max_8h_avg'].notna().sum():,} / {len(daily):,} station-days")

Daily station-day rows: 46,680
Daily date range: 2014-05-13 to 2025-01-04
Daily means available after the 20-hour rule:
  PM2.5: 43,026 / 46,680 station-days
  PM10: 33,957 / 46,680 station-days
  SO2: 43,525 / 46,680 station-days
  NO2: 43,469 / 46,680 station-days
  CO: 43,252 / 46,680 station-days
O3 daily 8-hour maximum available: 39,294 / 46,680 station-days


## 4. Align with the existing daily pollutant CSV

This cell loads `air-quality-classification/data/aqi_model_dataset.csv` and maps the new daily measurements to its pollutant-only feature names. The daily mean definitions and validity thresholds match the project’s AQI preparation rules; O3 maps to the daily maximum trailing 8-hour average. Both sources report CO in mg/m³, so no unit conversion is applied. The old file has no station ID, so its station names are mapped to IDs from the new source for matching.

The date ranges overlap. To avoid duplicate station-days and preserve the existing dataset values, this merge keeps the old row wherever both files have the same station and date. It drops target-label columns from the output because the new daily source has no corresponding target labels; the original old CSV remains unchanged. The measurement-count columns stay in the separate new daily pollutant CSV.

In [4]:
POLLUTANT_COLUMNS = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
OUTPUT_COLUMNS = ["date", "station", *POLLUTANT_COLUMNS]
NEW_TO_OLD_COLUMN_NAMES = {
    "PM2.5_24h_avg": "PM2.5",
    "PM10_24h_avg": "PM10",
    "SO2_24h_avg": "SO2",
    "NO2_24h_avg": "NO2",
    "CO_24h_avg": "CO",
    "O3_daily_max_8h_avg": "O3",
}

legacy_full = pd.read_csv(
    LEGACY_DAILY_INPUT,
    dtype={"station": "string"},
    low_memory=False,
)
required_legacy_columns = ["date", "station", *POLLUTANT_COLUMNS]
missing_legacy_columns = [column for column in required_legacy_columns if column not in legacy_full.columns]
if missing_legacy_columns:
    raise ValueError(f"Legacy daily pollutant file is missing columns: {missing_legacy_columns}")
legacy = legacy_full[required_legacy_columns].copy()
legacy["date"] = pd.to_datetime(legacy["date"], errors="coerce").dt.strftime("%Y-%m-%d")
if legacy["date"].isna().any():
    raise ValueError("The legacy daily pollutant file contains invalid dates.")
legacy["station"] = legacy["station"].astype("string").str.strip()
for pollutant in POLLUTANT_COLUMNS:
    legacy[pollutant] = pd.to_numeric(legacy[pollutant], errors="coerce")

# Build a one-to-one station-name-to-ID lookup so deduplication uses station IDs.
station_map = hourly[["station_id", "station_name"]].drop_duplicates().copy()
station_map["station_key"] = station_map["station_name"].astype("string").str.strip().str.casefold()
if station_map["station_key"].duplicated().any() or station_map["station_id"].duplicated().any():
    raise ValueError("The station name / ID mapping is not one-to-one.")
station_id_by_name = dict(zip(station_map["station_key"], station_map["station_id"]))
legacy["_station_key"] = legacy["station"].astype("string").str.strip().str.casefold()
legacy["_station_id"] = legacy["_station_key"].map(station_id_by_name)
if legacy["_station_id"].isna().any():
    unknown = sorted(legacy.loc[legacy["_station_id"].isna(), "station"].dropna().unique().tolist())
    raise ValueError(f"Legacy station names could not be mapped to IDs: {unknown}")

new_to_append = daily[
    ["date", "station_id", "station_name", *NEW_TO_OLD_COLUMN_NAMES.keys()]
].rename(
    columns={
        "station_id": "_station_id",
        "station_name": "station",
        **NEW_TO_OLD_COLUMN_NAMES,
    }
)
new_to_append = new_to_append[["date", "station", "_station_id", *POLLUTANT_COLUMNS]].copy()
new_to_append["date"] = pd.to_datetime(new_to_append["date"], errors="coerce").dt.strftime("%Y-%m-%d")
new_to_append["station"] = new_to_append["station"].astype("string").str.strip()
new_to_append["_station_key"] = new_to_append["station"].astype("string").str.strip().str.casefold()
for pollutant in POLLUTANT_COLUMNS:
    new_to_append[pollutant] = pd.to_numeric(new_to_append[pollutant], errors="coerce")

legacy["_source_priority"] = 0
new_to_append["_source_priority"] = 1
if legacy.duplicated(["date", "_station_id"]).any():
    raise ValueError("The legacy daily pollutant file contains duplicate station-days.")
if new_to_append.duplicated(["date", "_station_id"]).any():
    raise ValueError("The new daily pollutant file contains duplicate station-days.")

legacy_keys = set(zip(legacy["date"], legacy["_station_id"]))
new_keys = set(zip(new_to_append["date"], new_to_append["_station_id"]))
overlap_count = len(legacy_keys & new_keys)
new_only_count = len(new_keys - legacy_keys)

# Compare pollutant values on overlapping station-days before choosing the old rows.
overlap = legacy[["date", "_station_id", *POLLUTANT_COLUMNS]].merge(
    new_to_append[["date", "_station_id", *POLLUTANT_COLUMNS]],
    on=["date", "_station_id"],
    how="inner",
    suffixes=("_old", "_new"),
    validate="one_to_one",
)
print(f"Overlapping station-days: {overlap_count:,}; old values will be retained for these rows.")
print("Overlap comparison (paired readings only): median absolute difference by pollutant")
for pollutant in POLLUTANT_COLUMNS:
    old_values = pd.to_numeric(overlap[f"{pollutant}_old"], errors="coerce")
    new_values = pd.to_numeric(overlap[f"{pollutant}_new"], errors="coerce")
    paired = old_values.notna() & new_values.notna()
    median_difference = (old_values[paired] - new_values[paired]).abs().median()
    print(f"  {pollutant}: {int(paired.sum()):,} paired; median absolute difference {median_difference:.4g}")

combined_rows = pd.concat(
    [
        legacy[["date", "station", *POLLUTANT_COLUMNS, "_station_id", "_source_priority"]],
        new_to_append[["date", "station", *POLLUTANT_COLUMNS, "_station_id", "_source_priority"]],
    ],
    ignore_index=True,
)
combined_daily = (
    combined_rows.sort_values(["_source_priority", "date", "_station_id"])
    .drop_duplicates(["date", "_station_id"], keep="first")
    .drop(columns=["_station_id", "_source_priority"])
    .loc[:, OUTPUT_COLUMNS]
    .sort_values(["date", "station"])
    .reset_index(drop=True)
)
if combined_daily.duplicated(["date", "station"]).any():
    raise ValueError("Duplicate date/station rows remain after merging.")
if list(combined_daily.columns) != OUTPUT_COLUMNS:
    raise ValueError("The combined file does not match the intended pollutant-only schema.")

print(f"Old rows: {len(legacy):,}")
print(f"New rows available: {len(new_to_append):,}")
print(f"New station-days added after deduplication: {new_only_count:,}")
print(f"Combined pollutant rows: {len(combined_daily):,}")

Overlapping station-days: 12,171; old values will be retained for these rows.
Overlap comparison (paired readings only): median absolute difference by pollutant
  PM2.5: 10,610 paired; median absolute difference 2.801
  PM10: 4,793 paired; median absolute difference 3.896
  SO2: 10,694 paired; median absolute difference 0.5375
  NO2: 10,624 paired; median absolute difference 3.398
  CO: 10,669 paired; median absolute difference 0.08788
  O3: 10,254 paired; median absolute difference 6.631
Old rows: 17,376
New rows available: 46,680
New station-days added after deduplication: 34,509
Combined pollutant rows: 51,885


## 5. Save the daily pollutant files

This cell saves the newly aggregated pollutant data and the separate old-plus-new pollutant-only CSV. The old source file is not overwritten. The merged output uses the old pollutant feature names and column order (`date`, `station`, and six pollutants); target labels and weather fields are not included.

In [5]:
DAILY_POLLUTANT_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
COMBINED_POLLUTANTS_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
daily.to_csv(DAILY_POLLUTANT_OUTPUT, index=False, encoding="utf-8")
combined_daily.to_csv(COMBINED_POLLUTANTS_OUTPUT, index=False, encoding="utf-8")

for output_path in [DAILY_POLLUTANT_OUTPUT, COMBINED_POLLUTANTS_OUTPUT]:
    size_mib = output_path.stat().st_size / (1024 ** 2)
    print(f"Saved {output_path} ({size_mib:.2f} MiB)")

Saved C:\Users\PLP\Desktop\DataScienceProjects\Data_Science\CNEMC_Figshare_2014-2025\data\beijing_project_stations_daily.csv (6.71 MiB)
Saved C:\Users\PLP\Desktop\DataScienceProjects\Data_Science\air-quality-classification\data\beijing_daily_pollutants_combined_2013-2025.csv (5.16 MiB)


## 6. Review the merged pollutant data

This final cell confirms the combined row count, station and date coverage, and duplicate check, then prints a small sample. New-only rows have pollutant values but no old-model target labels because this output is intentionally pollutant-only.

In [6]:
print(f"Combined date range: {combined_daily['date'].min()} to {combined_daily['date'].max()}")
print(f"Stations in combined data: {combined_daily['station'].nunique()}")
print(f"Rows: {len(combined_daily):,}; duplicate station-days: {int(combined_daily.duplicated(['date', 'station']).sum())}")
print(f"Columns: {list(combined_daily.columns)}")
print("Sample rows:")
print(combined_daily.head(5).to_string(index=False))

Combined date range: 2013-03-01 to 2025-01-04
Stations in combined data: 12
Rows: 51,885; duplicate station-days: 0
Columns: ['date', 'station', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3']
Sample rows:
      date      station    PM2.5      PM10       SO2       NO2       CO     O3
2013-03-01 Aotizhongxin 7.125000 10.750000 11.708333 22.583333 0.429167 74.500
2013-03-01    Changping 5.083333 18.958333 16.041667 15.333333 0.387500 88.875
2013-03-01     Dingling 6.375000  6.270833  3.000000  2.333333 0.204167 86.000
2013-03-01       Dongsi 6.416667  9.875000  8.291667 28.229167 0.389583 88.000
2013-03-01     Guanyuan 7.541667 11.666667  8.500000 28.500000 0.400000 74.000
